In [ ]:
# Imports
import requests
import pandas as pd
import time
import random
from bs4 import BeautifulSoup

In [ ]:
# Seed keywords (English, target audience = Western EU/US) + locales
seed_keywords = [
    "lithium Bosnia",
    "Lopare lithium mine",
    "Bosnia mining",
    "Bosnia natural resources",
    "bauxite Bosnia",
    "coal mining Bosnia",
    "hydropower Bosnia",
    "Bosnia forests",
    "Bosnia water resources",
    "rare earth minerals Balkans",
    "Bosnia and Herzegovina minerals",
    "Bosnia mining investment",
    "lithium mining Europe",
    "Balkan natural resources",
    "Bosnia iron ore",
    "Zenica steel Bosnia",
    "Bosnia zinc mining",
    "Bosnia lead mining",
    "Tuzla salt mines",
    "Bosnia timber industry",
    "Bosnia wood export",
    "Bosnia wind energy",
    "Bosnia solar energy",
    "Bosnia natural gas",
    "Bosnia oil reserves",
    "Bosnia copper mining",
    "Bosnia manganese",
    "Bosnia agriculture land",
]

locales = [
    {"gl": "us", "hl": "en"},  # USA
    {"gl": "gb", "hl": "en"},  # UK
    {"gl": "de", "hl": "en"},  # Germany
    {"gl": "at", "hl": "en"},  # Austria
    {"gl": "fr", "hl": "en"},  # France
    {"gl": "it", "hl": "en"},  # Italy
    {"gl": "nl", "hl": "en"},  # Netherlands
]

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

In [ ]:
# Google Autocomplete scraper
def get_autocomplete_suggestions(query, gl="us", hl="en"):
    url = "http://suggestqueries.google.com/complete/search"
    params = {"client": "firefox", "q": query, "gl": gl, "hl": hl}
    try:
        r = requests.get(url, params=params, headers=HEADERS, timeout=5)
        r.raise_for_status()
        return r.json()[1]
    except Exception as e:
        print(f"[Autocomplete] Error for '{query}' ({gl}): {e}")
        return []

In [ ]:
# Related searches via SerpApi (with graceful fallback)
import os
SERPAPI_KEY = os.environ.get("SERPAPI_KEY", "")  # set this in your .env file, otherwise leave empty
def get_related_searches(query, gl="us", hl="en"):
    if not SERPAPI_KEY or not SERPAPI_KEY.strip():
        return []  # no key provided, skip silently

    url = "https://serpapi.com/search"
    params = {
        "q": query,
        "gl": gl,
        "hl": hl,
        "api_key": SERPAPI_KEY,
    }
    try:
        r = requests.get(url, params=params, timeout=20)

        if r.status_code == 401:
            print("[RelatedSearches] Invalid API key - skipping related searches.")
            return []

        r.raise_for_status()
        data = r.json()

        if "error" in data:
            print(f"[RelatedSearches] API error: {data['error']} - skipping.")
            return []

        related = data.get("related_searches", [])
        return [item["query"] for item in related]
    except Exception as e:
        print(f"[RelatedSearches] Error for '{query}' ({gl}): {e} - skipping.")
        return []

In [ ]:
# Run scraper across seeds and locales
rows = []

for seed in seed_keywords:
    for loc in locales:
        # Autocomplete
        for s in get_autocomplete_suggestions(seed, gl=loc["gl"], hl=loc["hl"]):
            rows.append({
                "seed_keyword": seed,
                "suggested_keyword": s,
                "source": "autocomplete",
                "country_sim": loc["gl"]
            })

        time.sleep(random.uniform(1.0, 2.0))

        # Related searches
        for s in get_related_searches(seed, gl=loc["gl"], hl=loc["hl"]):
            rows.append({
                "seed_keyword": seed,
                "suggested_keyword": s,
                "source": "related_search",
                "country_sim": loc["gl"]
            })

        time.sleep(random.uniform(1.5, 3.0))

print(f"Collected {len(rows)} rows")

Collected 1447 rows


In [ ]:
# Build DataFrame, clean, save
df = pd.DataFrame(rows)
df_unique = df.drop_duplicates(subset="suggested_keyword").reset_index(drop=True)

print(f"Total raw rows: {len(df)}")
print(f"Unique keywords: {len(df_unique)}")
print(df_unique["source"].value_counts())

output_path = "keywords_natural_resources_bih.csv"
df_unique.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

df_unique.head(20)

Total raw rows: 1447
Unique keywords: 306
source
related_search    219
autocomplete       87
Name: count, dtype: int64
Saved to keywords_natural_resources_bih.csv


,seed_keyword,suggested_keyword,source,country_sim
0,lithium Bosnia,lithium bosnia,autocomplete,us
1,lithium Bosnia,bosnia lithium mining,autocomplete,us
2,lithium Bosnia,lithium in bosnia and herzegovina,autocomplete,us
3,lithium Bosnia,arcore lithium bosnia,autocomplete,us
4,lithium Bosnia,what country has the most lithium,autocomplete,us
5,lithium Bosnia,biggest lithium countries,autocomplete,us
6,lithium Bosnia,low lithium levels while taking lithium,autocomplete,us
7,lithium Bosnia,Lithium bosnia map,related_search,us
8,lithium Bosnia,Lithium bosnia lopare,related_search,us
9,lithium Bosnia,Lithium bosnia news,related_search,us


In [ ]:
# List all unique suggested keywords to identify off-topic/junk keywords for cleanup step
for kw in df_unique["suggested_keyword"]:
  print(kw)

lithium bosnia
bosnia lithium mining
lithium in bosnia and herzegovina
arcore lithium bosnia
what country has the most lithium
biggest lithium countries
low lithium levels while taking lithium
Lithium bosnia map
Lithium bosnia lopare
Lithium bosnia news
Rio Tinto
Arcore AG
lopare lithium mine
lithium mine locations
lithium mine process
lithium biggest mine
Rock Tech Lithium
Lopare lithium mine news
bosnia mining
bosnia mining jobs
bosnia mining companies
bosnia mining industry
bosnia mining landslide
bosnia and herzegovina mining
adriatic mining bosnia
bitcoin mining bosnia
coal mining bosnia
Bosnia mining companies
Bosnia mining news
Bosnia mining map
Bosnia landmines map 2025
Bosnia mining 2021
Bosnia landmines 2024
Why are there so many landmines in Bosnia
Bosnia mine map app
Bosnia mining disaster
Who put landmines in bosnia
Bosnia landmine deaths per year
Bosnia landmines map
Where are landmines in bosnia
Bosnia mining rig
bosnia natural resources
bosnia herzegovina natural resour

In [ ]:
# Cleanup - remove off-topic/junk keywords
junk_keywords = [
    # Landmines / unrelated mining tangents
    "Bosnia landmines map 2025",
    "Bosnia landmines 2024",
    "Why are there so many landmines in Bosnia",
    "Bosnia mine map app",
    "Bosnia mining disaster",
    "Who put landmines in bosnia",
    "Bosnia landmine deaths per year",
    "Bosnia landmines map",
    "Where are landmines in bosnia",
    "Bosnia mining rig",
    "Bosnia copper mining rig",
    "bitcoin mining bosnia",
    "low lithium levels while taking lithium",

    # General country facts / politics / economy (not natural resources)
    "is bosnia a third world country",
    "is bosnia a poor country",
    "national food of bosnia",
    "Where is Bosnia currency",
    "Bosnia and herzegovina border countries",
    "Bosnia is located in which country",
    "Bosnia and Herzegovina",
    "is bosnia a democratic country",
    "is bosnia a democracy",
    "is bosnia a good country",
    "is bosnia a free country",
    "bosnia tax rate",
    "bosnia and herzegovina tax rate",
    "how wealthy is bosnia and herzegovina",
    "taxes in bosnia",
    "bosnia median income",
    "bosnia under which country",
    "is bosnia in the balkans",
    "Bosnia and Herzegovina travel guide",
    "Republic of Bosnia and Herzegovina",
    "Bosnija",
    "Bosnia and Herzegovina Map",
    "History of Bosnia",
    "Bosnia today",
    "Mostar wiki",
    "Bosnia religion",
    "Bosnia football",
    "Bosnia currency",
    "Bosnia map",
    "Bosnia and Herzegovina vs",
    "Bosnia people",
    "Bosnia in Europe",
    "Bosnia and herzegovina trade",
    "Us trade with bosnia",
    "Bosnian economy",

    # Balkans general / unrelated
    "what is balkan",
    "are balkan countries safe",
    "Balkan countries",
    "How many Balkan countries are there",
    "Why is it called the Balkans",
    "Balkan people",
    "What are the 11 Balkan countries",
    "Balkan natural resources albania",
    "List of balkan natural resources",
    "Balkan natural resources notes",

    # Rare earths - too generic/global, not B&H specific
    "Rare earth minerals balkans wikipedia",
    "Rare earth minerals balkans list",
    "Rare earth minerals sweden 2025",
    "Rare earth mining by country",
    "Australian reserves of rare earth minerals",
    "How to find rare earth minerals",
    "Rare earth in arctic",
    "Huge rare earth metals discovery in arctic sweden",
    "Yttrium deposits by country",
    "Is china the only country with rare earth minerals",
    "European rare earth companies",
    "Rare earths europe map",
    "Rare earth elements",
    "Rare earth magnets and motors a european call for action",
    "Critical raw materials Act",

    # Lithium - too generic/global, not B&H specific
    "what country has the most lithium",
    "biggest lithium countries",
    "which country has the most lithium mines",
    "Lithium production by country",
    "Lithium reserves by country",
    "European Lithium stock",
    "Largest lithium deposits in europe",
    "Europe lithium demand",
    "Portugal lithium reserves",
    "European Lithium chart",
    "European Lithium Ltd",
    "Who owns European Lithium",
    "European Lithium stock ASX",
    "Lithium mining europe pdf",
    "Is european lithium a good investment",
    "European Lithium announcements",
    "Lithium supply chains",
    "Lithium commodity",
    "Lithium market supply and demand",
    "Lithium demand and supply forecast",
    "Lithium mining portugal map",
    "coal mining by country",
    "which countries produce the most hydroelectric power",
    "hydropower capacity by country",

    # Forests - off topic
    "Bosnia forests weather",
    "Bosnia forests holidays",
    "Bosnia forests animals",
    "Primeval forests in Europe",
    "Is perucica a rainforest",
    "Bosnia water resources phone number",
    "Bosnia water resources wikipedia",

    # Zenica/steel - mostly off-topic geography
    "Zenica steel bosnia address",
    "Zenica bosnia and herzegovina",
    "Why does bosnia play in zenica",
    "Zenica bosnia currency",
    "Where is Zenica in which country",
    "Zenica population",
    "Zenica map",
    "Zenica Bosnia",
    "Zenica religion",
    "Zenica Bosnia demographics",
    "ArcelorMittal owner",

    # Tuzla salt - off-topic tangents
    "what are salt mines used for",
    "are salt mines safe",
    "what are salt mines",
    "Map of Sarajevo",
    "Municipalities of bosnia",
    "Hac city",
    "Sarajevo mountains",
    "Sarajevo summer",
    "Facts about mostar",
    "Tuzlanska kapija",
    "Tuzla salt mines tours",
    "Tuzla salt mines wikipedia",

    # Solar - off-topic
    "solar energy degree programs",
    "solar energy which country",
    "Future prospects of renewable energy",

    # Oil - too generic/global comparisons
    "does russia have oil reserves",
    "does brazil have oil reserves",
    "which country has the smallest oil reserves",
    "how long will russia's oil reserves last",
    "European oil fields",

    # Agriculture land - real estate listings, off-topic
    "Bosnia agriculture land for sale",
    "Bosnia agriculture land price",
    "Bosnia agriculture land prices",
    "Bosnia agriculture land nekretnine",
    "Land for sale in Bosnia",
    "Farm house for sale in bosnia",
    "Bosnia agriculture land jobs",
]

df_clean = df_unique[~df_unique["suggested_keyword"].isin(junk_keywords)].reset_index(drop=True)

print(f"Before cleanup: {len(df_unique)}")
print(f"After cleanup: {len(df_clean)}")

df_clean.to_csv("keywords_natural_resources_bih.csv", index=False)
print("Saved cleaned CSV")
df_clean

Before cleanup: 306
After cleanup: 164
Saved cleaned CSV


,seed_keyword,suggested_keyword,source,country_sim
0,lithium Bosnia,lithium bosnia,autocomplete,us
1,lithium Bosnia,bosnia lithium mining,autocomplete,us
2,lithium Bosnia,lithium in bosnia and herzegovina,autocomplete,us
3,lithium Bosnia,arcore lithium bosnia,autocomplete,us
4,lithium Bosnia,Lithium bosnia map,related_search,us
...,...,...,...,...
159,Bosnia copper mining,Bosnia copper mining sarajevo,related_search,gb
160,Bosnia copper mining,Bosnia copper mining 2021,related_search,gb
161,Bosnia copper mining,Bosnia copper mining news,related_search,nl
162,Bosnia copper mining,Bosnia copper mining companies,related_search,nl
